# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library. It covers metadata review, record set extraction, data processing, and visualization.

### Dataset Source
The dataset is defined as a FAIR^2 Croissant schema, accessible at the URL below.

Schema URL:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata object (do NOT subscript or iterate)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Dataset Identifier: {metadata.identifier}")
print(f"Dataset Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and their @ids
record_sets = dataset.record_sets

print("Available record sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, Name: {rs.get('name', '(no name)')}")

# For demonstration, choose the first available record set
record_set_id = record_sets[0]['@id'] if record_sets else None

# List all fields for this record set
if record_set_id:
    fields = dataset.fields(record_set=record_set_id)
    print(f"\nFields for record set {record_set_id}:")
    for f in fields:
        print(f"  @id: {f['@id']}, Name: {f.get('name', '(no name)')} [{f.get('dataType', 'unknown type')}]")
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis.

You can use the listed record set and field `@id`s above to select and extract data. Here, we'll load all available record sets.

In [ ]:
# Extract data for all record sets
dataframes = {}

for rs in record_sets:
    rs_id = rs['@id']
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for record set {rs_id} with shape {df.shape}")

# Display columns from the primary record set (first record set)
if record_set_id:
    print("\nColumns in primary record set:")
    print(dataframes[record_set_id].columns.tolist())
    display(dataframes[record_set_id].head(3))

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping.

We'll select a numeric field for filtering and normalization.
(Replace the `@id` example with a true numeric field's `@id` from section 2.)

In [ ]:
from numpy import nan
import numpy as np

# Replace with an actual numeric field '@id' from previous output
# Example: 'https://api.app.sen.science/frontiers/7862866/age_field_id'
numeric_field_id = None

fields = dataset.fields(record_set=record_set_id)
# Attempt to find a numeric-type field
for f in fields:
    if f.get('dataType', '').lower() in ['integer', 'float', 'number']:
        numeric_field_id = f['@id']
        print(f"Using numeric field: {numeric_field_id} [{f.get('name')}]")
        break
if not numeric_field_id:
    print("No numeric field detected in record set. EDA demo will be limited.")

# Demo: filter, normalize, and group
df = dataframes.get(record_set_id, pd.DataFrame())

if numeric_field_id and numeric_field_id in df.columns:
    threshold = 10
    numeric_vals = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df.loc[numeric_vals > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    field_mean = numeric_vals.mean()
    field_std = numeric_vals.std()
    filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - field_mean) / field_std
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field
    # Search for a categorical/text field @id
    group_field_id = None
    for f in fields:
        if f.get('dataType', '').lower() in ['text', 'string']:
            group_field_id = f['@id']
            print(f"Using group field: {group_field_id} [{f.get('name')}]")
            break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Visualize distributions or relationships between key fields. Here, we plot the distribution of a numeric field and its mean per category (if available).

In [ ]:
import matplotlib.pyplot as plt

# Plot histogram for numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    plt.hist(pd.to_numeric(df[numeric_field_id], errors='coerce'), bins=15, color='skyblue', edgecolor='k')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Bar plot for group means (if grouped_df exists)
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(8, 4))
        plt.bar(grouped_df[group_field_id], grouped_df[numeric_field_id])
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} per {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
We demonstrated loading and exploration of the Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors dataset using `mlcroissant`. We reviewed metadata, record sets, and fields (referencing them by `@id` throughout), extracted tabular data for analysis, and performed basic EDA including normalization and grouping by key attributes.

For more advanced analyses, refer to relationships between fields and additional Croissant schema elements (using their `@id`s) as documented in the FAIR^2 schema.